# MRI (ACDC) Preprocessing

Mirrors `ecg_preprocessing.ipynb`'s structure for the MRI modality: audit the dataset, import the
`mri_preprocessing` module from `src/`, walk through the full pipeline on one example patient (with visual
sanity-checking of the cropped images), then apply it across the whole ACDC dataset (150 patients - small
enough to process in full, unlike PTB-XL's ~22k records) to get a dataset-wide numeric feature table.

There is no `mri_agent.py` yet (the MRI equivalent of `ecg_agent.py` - LLM-based structured findings), so
this notebook stops at preprocessing and numeric features, matching how `ecg_preprocessing.ipynb`'s
"structured workflow" section stops at `ecg_preprocessing_pipeline` before the agent step is introduced.


In [ ]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt


def find_project_root(start=None, marker="CLAUDE.md"):
    """Walk upward from `start` (default: this notebook's cwd) until a
    directory containing `marker` is found."""
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f"Could not locate project root (missing {marker}) from {current}")


PROJECT_ROOT = find_project_root()
ACDC_DIR = PROJECT_ROOT / "ACDC" / "database"

src_dir = PROJECT_ROOT / "src"
sys.path.insert(0, str(src_dir))

print("Project root:", PROJECT_ROOT)
print("ACDC directory:", ACDC_DIR)


## 1. Dataset audit

ACDC has no single metadata CSV like PTB-XL's `ptbxl_database.csv` - each patient is just a
`patientXXX/` directory under `training/` or `testing/`, with per-patient metadata in its own `Info.cfg`.


In [ ]:
training_patients = sorted(p.name for p in (ACDC_DIR / "training").iterdir() if p.name.startswith("patient"))
testing_patients = sorted(p.name for p in (ACDC_DIR / "testing").iterdir() if p.name.startswith("patient"))

print("Training patients:", len(training_patients))
print("Testing patients: ", len(testing_patients))
print("First 3 training patients:", training_patients[:3])


## 2. Import the preprocessing module

In [ ]:
import base64
import io

from PIL import Image

from mri_preprocessing import preprocess_patient

print("Module imported successfully.")
print("Available function:", preprocess_patient.__name__)


## 3. Single-patient walkthrough

Run the full pipeline on one example patient, inspect its numeric features, and visually display the
cropped ED/ES images (same ROI, same scale - see `union_bounding_boxes` in `mri_preprocessing.py`) before
trusting them for anything downstream.


In [ ]:
example_patient_dir = ACDC_DIR / "training" / "patient001"
result = preprocess_patient(str(example_patient_dir))

print("Patient:", result["patient_id"])
print("Info:", result["info"])
print("Slice indices (ED/ES):", result["slice_indices"])
print("Shared ROI bounding box:", result["roi_bbox"])
print()
print("Features:")
for key, value in result["features"].items():
    print(f"  {key}: {value}")


In [ ]:
ed_image = Image.open(io.BytesIO(base64.b64decode(result["images"]["short_axis_mid_ed"])))
es_image = Image.open(io.BytesIO(base64.b64decode(result["images"]["short_axis_mid_es"])))

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(ed_image, cmap="gray")
axes[0].set_title(f"{result['patient_id']} - ED (frame {result['info']['ED']})")
axes[0].axis("off")
axes[1].imshow(es_image, cmap="gray")
axes[1].set_title(f"{result['patient_id']} - ES (frame {result['info']['ES']})")
axes[1].axis("off")
plt.tight_layout()

output_dir = PROJECT_ROOT / "results" / "records" / result["patient_id"] / "mri_pipeline"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"{result['patient_id']}_ed_es_roi.png"
plt.savefig(output_path, dpi=150)
print("Figure saved to:", output_path)
plt.show()


## 3b. Generate the MRI agent''''s structured findings

Uses the free local Ollama backend by default (no Anthropic key needed), same as the ECG agent. Requires
Ollama installed and running locally with MedGemma pulled:
    ollama pull medgemma:4b


In [ ]:
from agent_common import save_contribution
from mri_agent import generate_mri_contribution

mri_contribution = generate_mri_contribution(result, image_mode="ed_only")
print(mri_contribution)

mri_output_dir = PROJECT_ROOT / "results" / "records" / result["patient_id"]
mri_output_path = save_contribution(mri_contribution, output_dir=str(mri_output_dir), file_name="mri_contribution")
print("Saved to:", mri_output_path)


## 4. Apply across the whole dataset

ACDC is small enough (150 patients) to process in full rather than a sample. Each patient is wrapped in a
try/except so one unreadable record doesn't stop the whole batch - failures are collected and reported
separately instead of silently skipped.


In [ ]:
all_patients = [(p, ACDC_DIR / "training" / p) for p in training_patients] + [
    (p, ACDC_DIR / "testing" / p) for p in testing_patients
]

rows = []
failures = []
for patient_id, patient_dir in all_patients:
    try:
        r = preprocess_patient(str(patient_dir))
        rows.append(
            {
                "patient_id": r["patient_id"],
                "group": r["info"]["Group"],
                "lv_volume_ed_ml": r["features"]["lv_volume_ed_ml"],
                "lv_volume_es_ml": r["features"]["lv_volume_es_ml"],
                "rv_volume_ed_ml": r["features"]["rv_volume_ed_ml"],
                "rv_volume_es_ml": r["features"]["rv_volume_es_ml"],
                "lv_ejection_fraction_pct": r["features"]["lv_ejection_fraction_pct"],
                "rv_ejection_fraction_pct": r["features"]["rv_ejection_fraction_pct"],
            }
        )
    except Exception as exc:
        failures.append((patient_id, str(exc)))

features_df = pd.DataFrame(rows)
print("Processed successfully:", len(features_df), "/", len(all_patients))
if failures:
    print("Failures:", failures)
features_df.head()


## 5. Dataset-wide sanity check by pathology group

Same cross-check used earlier on a handful of patients, now run over the entire dataset: mean LV ejection
fraction should be low for DCM, near-normal/high for HCM, intermediate for MINF, and normal for NOR.


In [ ]:
summary = features_df.groupby("group")["lv_ejection_fraction_pct"].agg(["mean", "std", "count"])
print(summary)

csv_path = PROJECT_ROOT / "results" / "mri_features_all_patients.csv"
features_df.to_csv(csv_path, index=False)
print()
print("Full feature table saved to:", csv_path)
